<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# Duckiedrone Docker Hosts and Stacks

The local exercises showed how to build an image, run a container, inspect its behavior, and manage persistent data. On a Duckiedrone, these same Docker concepts support a collection of cooperating services.

For example, making camera images available to a [Robot Operating System 2 (__ROS 2__)](https://www.ros.org/) application involves more than a single container. One service reads the camera, another carries its messages, and a bridge makes those messages available to ROS 2.

Before inspecting such a system, we need to answer two questions: __which Docker host runs the service, and which group of services does it belong to?__

This notebook connects the local exercises to the Duckiedrone's software organization. You will distinguish the base-station and Duckiedrone environments, identify the default stacks, and follow a sensor message through several containers.

## Two Docker hosts

A __Docker host__ is the environment running the Docker daemon and its containers.

For a physical Duckiedrone, the base station and the Duckiedrone have separate Docker daemons. Building `lx-docker-hello:0.1` on your base station does not make that image available on the Duckiedrone automatically.

Likewise, listing containers through the base station's local daemon does not show the containers managed by the physical Duckiedrone's daemon. [Table 1](#table-1) compares the resources managed by these two Docker hosts.

<table id="table-1" class="lx-table">
  <caption>Table 1: Docker resources on the base station and a physical Duckiedrone.</caption>
  <thead>
    <tr>
      <th>Docker host</th>
      <th>Resources it may manage</th>
      <th>Management in this LX</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Base station</td>
      <td>Practice images, containers, and volumes; development tools; virtual Duckiedrone services</td>
      <td>Modify only the resources you created for the local exercises</td>
    </tr>
    <tr>
      <td>Physical Duckiedrone</td>
      <td>Sensor, communication, dashboard, and flight-support services on its embedded computer</td>
      <td>Use the documented Duckietown workflows to manage the Duckiedrone</td>
    </tr>
  </tbody>
</table>

The location of the terminal still does not establish the target. A client running on the base station can send requests to a remote daemon. [Figure 1](#figure-1) shows how the client's connection selects the Docker host.

<figure id="figure-1" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    base-station terminal
      |
      | Docker client connection
      |
      +-> local daemon; practice containers and other local services
      `-> physical Duckiedrone daemon; Duckiedrone services
  </pre>
  <figcaption>
    Figure 1: One client location can provide access to different Docker hosts.
  </figcaption>
</figure>

Suppose you successfully built the practice web-server image, but cannot find it while inspecting a physical Duckiedrone. That does not establish a failed build. First check which daemon's image inventory you are viewing: the image was created on the base station, so it belongs to that daemon's image inventory.

The same reasoning applies to container logs. To inspect a particular container's logs, your client must reach the daemon managing that container.

## Why group containers into stacks?

In the web-server exercise, one `docker run` command supplied the container name, image, and port mapping.

Now consider a system with sensor drivers, messaging services, bridges, and a dashboard. Each service needs the correct image and configuration. Reconstructing every startup command by hand would be difficult to maintain.

__Docker Compose__ describes services and their configuration in a YAML (YAML Ain't Markup Language) file. It can record image references, environment variables, mounts, networking, and other settings together. See [How Compose works](https://docs.docker.com/compose/intro/compose-application-model/).

In Duckietown documentation, these Compose-based groups are called __stacks__. A stack is a deployment grouping, not another container or another Docker host. Several stacks can run on one host, and services in different stacks can cooperate.

For example, the camera driver and its ROS 2 bridge (a connector transferring data from the driver to ROS 2) belong to different stacks, even though both contribute to the camera-data workflow.

## Duckiedrone stacks

The standard Duckiedrone update workflow selects the three stacks listed in [Table 2](#table-2).

<table id="table-2" class="lx-table">
  <caption>Table 2: Default Duckiedrone stacks and representative containers.</caption>
  <thead>
    <tr>
      <th>Stack</th>
      <th>Examples of containers</th>
      <th>Responsibility</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>robot/basics</code></td>
      <td><code>portainer</code>, <code>kvstore</code></td>
      <td>Container management interface and storage for small persistent settings</td>
    </tr>
    <tr>
      <td><code>duckietown/duckiedrone</code></td>
      <td><code>dtps</code>, <code>driver-camera</code>, <code>driver-tof-bottom</code>, <code>dashboard</code></td>
      <td>Sensor drivers, <a href="https://docs.duckietown.com/ente/duckietown-manual/04-software-tools/duckietown-postal-service-dtps.html?highlight=dtps">Duckietown Postal Service (DTPS)</a> messaging, and device services</td>
    </tr>
    <tr>
      <td><code>ros2/duckiedrone</code></td>
      <td><code>zenoh-router</code>, <code>ros2-camera</code>, <code>ros2-tof-bottom</code>, <code>ros2-mavros</code></td>
      <td>ROS 2 transport and bridges to sensor data and the flight controller</td>
    </tr>
  </tbody>
</table>

When updating the software of a Duckiedrone with `dts duckiebot update DUCKIEDRONE_NAME`, where `DUCKIEDRONE_NAME` is the robot’s name, what is happening under the hood is obtaining the required images and bringing up the default stacks. The Duckietown manual ["Preparing the software stack"](https://docs.duckietown.com/ente/opmanual-dd24/40-duckiedrone-handling/environment-setup.html#update-the-duckiedrone-software) page lists the services available after updating the basics stack.

The table lists representative containers rather than the complete inventory. Compare an actual deployment with the current [Duckiedrone container reference](https://docs.duckietown.com/ente/opmanual-dd24/60-software-architecture/containers-running-on-the-duckiedrone.html), taking its software version and configuration into account.

## Optional stacks

[Table 3](#table-3) lists stacks started only for particular activities.

| Stack | Purpose |
| --- | --- |
| `duckiedrone/extra_sensors` | Drivers and ROS 2 bridges for the front, left, right, and top ToF sensors |
| `duckiedrone/extra_tools` | Additional diagnostics and investigation tools |
| `ros2-core/duckiedrone` | A separate stack containing the `altitude` service |

These are not part of the standard three-stack update set. Their absence at startup is therefore not automatically a fault, rather a feature. Each service consumes computational resources, and to make sure the limited capacity of the onboard computer is not saturated when performing specific tasks, the optional stacks are started only when needed.

For example, finding `driver-tof-bottom` without `driver-tof-front` can be consistent with the default configuration: the bottom time of flight sensor is used to measure the robot's altitude, while the front sensor is only used for obstacle avoidance. If the optional sensor stack was not requested, the front sensor driver will not be running.

Use the relevant activity instructions for starting optional services. For example `dts stack up -H DUCKIEDRONE_NAME -d duckiedrone/extra_sensors`, where `-H` selects the Duckiedrone and `-d` runs the stack in the background, can be used to start the `extra_sensors` stack. The [container reference](https://docs.duckietown.com/ente/opmanual-dd24/60-software-architecture/containers-running-on-the-duckiedrone.html) distinguishes default, optional, separate, and legacy stacks, and provides an inventory of containers in each stack. 

## Summarizing the command locations

We have introduced Docker containers and stacks, running on different platforms, for different purposes. To summarize, [Table 1](#table-1) shows where you should enter each command and which computer's Docker resources it uses. These development boundaries are important to keep in mind when working with a physical Duckiedrone.

<table id="table-1" class="lx-table">
  <caption>Table 1: Command locations and Docker targets for common tasks.</caption>
  <thead>
    <tr>
      <th>Task</th>
      <th>Terminal</th>
      <th>Docker host</th>
      <th>Command or instructions</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Practice images, containers, ports, and volumes</td>
      <td>Base-station terminal</td>
      <td>Base station</td>
      <td>Use the local Docker context and the <code>lx-docker-*</code> resources in these exercises</td>
    </tr>
    <tr>
      <td>Update Duckiedrone software</td>
      <td>Base-station terminal</td>
      <td>Duckiedrone</td>
      <td><code>dts duckiebot update DUCKIEDRONE_NAME</code>, following the Duckiedrone manual's software update procedure</td>
    </tr>
    <tr>
      <td>Start optional Duckiedrone sensor services</td>
      <td>Base-station terminal</td>
      <td>Duckiedrone</td>
      <td><code>dts stack up -H DUCKIEDRONE_NAME -d duckiedrone/extra_sensors</code>, following the container reference</td>
    </tr>
    <tr>
      <td>Inspect Duckiedrone images and containers</td>
      <td>SSH shell on the Duckiedrone</td>
      <td>Duckiedrone</td>
      <td><code>docker image ls</code> and <code>docker container ls --all</code></td>
    </tr>
  </tbody>
</table>

## Final reminder: not all code must run in Docker containers

While Docker is a very useful technology, it is to be noted that not every part of a physical Duckiedrone’s flight system runs in Docker. For example, the flight controller software [Open Source Autopilot](https://px4.io/) (**PX4**) runs on the physical flight controller and performs vehicle attitude estimation and control, as well as other lower level functions. The `ros2-mavros` container then connects that system to ROS 2, exposing telemetry and command interfaces.

Consequently, when looking a the Docker containers running on a Duckiedrone, or any other robot, keep in mind that the container inventory does not need to describe every process involved in robot's operation. 

## Reflection

Before running the self-check, consider these questions:

1. Does building an image on the base station automatically make it available on a physical Duckiedrone?

2. Can a base-station terminal control a Docker daemon running elsewhere?

3. What does a Compose stack describe?

4. Why are `driver-camera` and `ros2-camera` separate services?

5. Does a missing `driver-tof-front` container necessarily indicate a failed default deployment?

6. On a physical Duckiedrone, does the `ros2-mavros` container replace the PX4 flight controller?

## Further reading

The DD24 manual's [software update instructions](https://docs.duckietown.com/ente/opmanual-dd24/40-duckiedrone-handling/10-software-update.html) describe the deployment workflow. Its [container reference](https://docs.duckietown.com/ente/opmanual-dd24/60-software-architecture/containers-running-on-the-duckiedrone.html) lists the stacks, and the [architecture guide](https://docs.duckietown.com/ente/opmanual-dd24/60-software-architecture/architecture-diagram.html) explains their connections to sensors and the flight controller.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()